# 01 - Preprocessing & Quality Control

This notebook prepares the PPG + IMU recordings for model development and performs
consistent quality-control checks for each session.

**Main pipeline**
1. Load cleaned sparse sensor recordings.
2. Split PPG, accelerometer and chest-reference streams.
3. Synchronize to a common 12.5 Hz (80 ms) time grid.
4. Use short-gap linear interpolation only:
   - PPG: maximum 240 ms gap
   - ACC: maximum 160 ms gap
5. Create non-overlapping 6-second windows (75 timesteps).
6. Use mean Chest HR within the same window as regression target.
7. Save the same QC figures for every session.

> Note: The interpolation thresholds are provisional until final supervisor confirmation.

## 1. Imports & Paths

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import preprocess_session

DATA_RAW = PROJECT_ROOT / "data" / "raw"
FIGURES = PROJECT_ROOT / "results" / "figures"
METRICS = PROJECT_ROOT / "results" / "metrics"

for folder in [
    FIGURES / "qc" / "sitting",
    FIGURES / "qc" / "bike",
    FIGURES / "qc" / "walking",
    FIGURES / "qc" / "food_preparation",
    FIGURES / "qc" / "writing",
    FIGURES / "qc" / "eating",
    FIGURES / "comparisons",
    FIGURES / "models",
    METRICS,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 2. QC Helper Functions

In [ ]:
GRID_MS = 80
PPG_MAX_GAP_MS = 240
ACC_MAX_GAP_MS = 160
WINDOW_SIZE = 75
WINDOW_SECONDS = 6


def _numeric_stream(df, cols):
    out = df[["Timestamp_ms"] + cols].copy()
    out["Timestamp_ms"] = pd.to_numeric(out["Timestamp_ms"], errors="coerce")
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna(subset=["Timestamp_ms"] + cols)
    out = out.sort_values("Timestamp_ms").drop_duplicates("Timestamp_ms")
    return out


def _controlled_interpolate(t_source, values, t_grid, max_gap_ms):
    # Linear interpolation only between two real neighboring samples
    # when their gap is not larger than max_gap_ms.
    t_source = np.asarray(t_source, dtype=float)
    values = np.asarray(values, dtype=float)
    t_grid = np.asarray(t_grid, dtype=float)

    result = np.full(len(t_grid), np.nan, dtype=float)

    for i, t in enumerate(t_grid):
        r = np.searchsorted(t_source, t, side="left")

        if r < len(t_source) and t_source[r] == t:
            result[i] = values[r]
            continue

        if r == 0 or r >= len(t_source):
            continue

        l = r - 1
        gap = t_source[r] - t_source[l]

        if gap <= max_gap_ms:
            frac = (t - t_source[l]) / gap
            result[i] = values[l] + frac * (values[r] - values[l])

    return result


def build_qc_sync(file_path, sheet_name):
    # Rebuild synchronized signals only for QC plotting.
    # Model-ready windows still come from src.preprocessing.preprocess_session().
    df = pd.read_excel(file_path, sheet_name=sheet_name)

    ppg = _numeric_stream(df, ["PPG_Raw"])
    acc = _numeric_stream(df, ["ACC_X", "ACC_Y", "ACC_Z"])
    chest = _numeric_stream(df, ["Chest_HR_BPM"])

    start_ms = max(
        ppg["Timestamp_ms"].min(),
        acc["Timestamp_ms"].min(),
        chest["Timestamp_ms"].min(),
    )
    end_ms = min(
        ppg["Timestamp_ms"].max(),
        acc["Timestamp_ms"].max(),
        chest["Timestamp_ms"].max(),
    )

    grid = np.arange(start_ms, end_ms + 1, GRID_MS)

    sync = pd.DataFrame({"Timestamp_ms": grid})

    sync["PPG_Raw"] = _controlled_interpolate(
        ppg["Timestamp_ms"].values,
        ppg["PPG_Raw"].values,
        grid,
        PPG_MAX_GAP_MS,
    )

    for axis in ["ACC_X", "ACC_Y", "ACC_Z"]:
        sync[axis] = _controlled_interpolate(
            acc["Timestamp_ms"].values,
            acc[axis].values,
            grid,
            ACC_MAX_GAP_MS,
        )

    return df, ppg, acc, chest, sync


def movement_intensity_per_window(X_fusion):
    values = []

    for window in X_fusion:
        acc_mag = np.sqrt(
            window[:, 1] ** 2 +
            window[:, 2] ** 2 +
            window[:, 3] ** 2
        )
        values.append(np.std(acc_mag))

    return np.asarray(values)


def save_session_qc(
    file_path,
    sheet_name,
    session_id,
    X_ppg,
    X_fusion,
    y,
    meta,
    figure_folder,
):
    figure_folder = Path(figure_folder)
    figure_folder.mkdir(parents=True, exist_ok=True)

    _, ppg, _, _, sync = build_qc_sync(file_path, sheet_name)

    # 1) Original vs resampled PPG
    valid_sync = sync.dropna(subset=["PPG_Raw"]).copy()

    if len(valid_sync) > 0:
        example_start = valid_sync["Timestamp_ms"].iloc[0]
        example_end = example_start + 10_000

        raw_seg = ppg[
            (ppg["Timestamp_ms"] >= example_start) &
            (ppg["Timestamp_ms"] <= example_end)
        ]

        sync_seg = valid_sync[
            (valid_sync["Timestamp_ms"] >= example_start) &
            (valid_sync["Timestamp_ms"] <= example_end)
        ]

        plt.figure(figsize=(12, 5))
        plt.plot(
            (raw_seg["Timestamp_ms"] - example_start) / 1000,
            raw_seg["PPG_Raw"],
            marker="o",
            markersize=2.5,
            linewidth=1,
            label="Original PPG",
        )
        plt.plot(
            (sync_seg["Timestamp_ms"] - example_start) / 1000,
            sync_seg["PPG_Raw"],
            linewidth=1.5,
            label="Resampled PPG",
        )
        plt.title(f"{session_id} - Original vs Resampled PPG")
        plt.xlabel("Time (seconds)")
        plt.ylabel("PPG Raw")
        plt.grid(alpha=0.2, linestyle="--")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            figure_folder / f"{session_id}_ppg_resampling.png",
            dpi=160,
            bbox_inches="tight",
        )
        plt.show()

    # 2) Chest HR over time
    time_min = (meta["Start_ms"].values - meta["Start_ms"].iloc[0]) / 60000

    plt.figure(figsize=(12, 5))
    plt.plot(time_min, y, linewidth=1.8, label="Chest HR")

    max_idx = int(np.argmax(y))
    plt.scatter(
        time_min[max_idx],
        y[max_idx],
        s=45,
        label="Maximum HR",
    )

    plt.title(f"{session_id} - Chest HR Over Time")
    plt.xlabel("Time (minutes)")
    plt.ylabel("Chest HR (BPM)")
    plt.grid(alpha=0.2, linestyle="--")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        figure_folder / f"{session_id}_chest_hr.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()

    # 3) Movement intensity
    movement = movement_intensity_per_window(X_fusion)

    plt.figure(figsize=(12, 5))
    plt.plot(time_min, movement, linewidth=1.8)

    max_movement_idx = int(np.argmax(movement))
    plt.scatter(
        time_min[max_movement_idx],
        movement[max_movement_idx],
        s=45,
        label="Highest movement",
    )

    plt.title(f"{session_id} - Movement Intensity")
    plt.xlabel("Time (minutes)")
    plt.ylabel("Std. of ACC Magnitude")
    plt.grid(alpha=0.2, linestyle="--")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        figure_folder / f"{session_id}_movement_intensity.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()

    # 4) Highest-movement ACC window
    highest = X_fusion[max_movement_idx]
    t = np.arange(len(highest)) * GRID_MS / 1000

    plt.figure(figsize=(12, 5))
    plt.plot(t, highest[:, 1], label="ACC_X")
    plt.plot(t, highest[:, 2], label="ACC_Y")
    plt.plot(t, highest[:, 3], label="ACC_Z")

    plt.title(f"{session_id} - ACC in Highest-Movement Window")
    plt.xlabel("Time within window (seconds)")
    plt.ylabel("Acceleration")
    plt.grid(alpha=0.2, linestyle="--")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        figure_folder / f"{session_id}_highest_movement_acc.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()

    print(f"{session_id}")
    print(f"  Valid windows : {len(y)}")
    print(f"  Mean HR       : {np.mean(y):.2f} BPM")
    print(f"  Median HR     : {np.median(y):.2f} BPM")
    print(f"  Min HR        : {np.min(y):.2f} BPM")
    print(f"  Max HR        : {np.max(y):.2f} BPM")
    print(f"  Std HR        : {np.std(y):.2f} BPM")
    print(f"  Max movement  : {np.max(movement):.5f}")

## 3. Sitting Sessions

### Sitting_01

In [ ]:
s01_path = DATA_RAW / "sitting" / "Sitting_Chest_Start_Cleaned.xlsx"

X_ppg_s01, X_fusion_s01, y_s01, meta_s01 = preprocess_session(
    s01_path,
    sheet_name="Sitting_Data",
    activity="Sitting",
    session_id="Sitting_01",
)

save_session_qc(
    s01_path,
    "Sitting_Data",
    "Sitting_01",
    X_ppg_s01,
    X_fusion_s01,
    y_s01,
    meta_s01,
    FIGURES / "qc" / "sitting",
)

### Sitting_02

In [ ]:
s02_path = DATA_RAW / "sitting" / "Sitting_02_Chest_Start_Cleaned.xlsx"

X_ppg_s02, X_fusion_s02, y_s02, meta_s02 = preprocess_session(
    s02_path,
    sheet_name="Sitting_Data",
    activity="Sitting",
    session_id="Sitting_02",
)

save_session_qc(
    s02_path,
    "Sitting_Data",
    "Sitting_02",
    X_ppg_s02,
    X_fusion_s02,
    y_s02,
    meta_s02,
    FIGURES / "qc" / "sitting",
)

### Sitting_03

In [ ]:
s03_path = DATA_RAW / "sitting" / "Sitting_03_Chest_Start_Cleaned.xlsx"

X_ppg_s03, X_fusion_s03, y_s03, meta_s03 = preprocess_session(
    s03_path,
    sheet_name="Sitting_Data",
    activity="Sitting",
    session_id="Sitting_03",
)

save_session_qc(
    s03_path,
    "Sitting_Data",
    "Sitting_03",
    X_ppg_s03,
    X_fusion_s03,
    y_s03,
    meta_s03,
    FIGURES / "qc" / "sitting",
)

### Sitting Summary

In [ ]:
sitting_summary = pd.DataFrame({
    "Session": ["Sitting_01", "Sitting_02", "Sitting_03"],
    "Windows": [len(y_s01), len(y_s02), len(y_s03)],
    "Mean_HR": [np.mean(y_s01), np.mean(y_s02), np.mean(y_s03)],
    "Median_HR": [np.median(y_s01), np.median(y_s02), np.median(y_s03)],
    "Min_HR": [np.min(y_s01), np.min(y_s02), np.min(y_s03)],
    "Max_HR": [np.max(y_s01), np.max(y_s02), np.max(y_s03)],
    "Std_HR": [np.std(y_s01), np.std(y_s02), np.std(y_s03)],
}).round(2)

sitting_summary.to_csv(METRICS / "sitting_summary.csv", index=False)
sitting_summary

## 4. Bike Sessions

### Bike_01

In [ ]:
b01_path = DATA_RAW / "bike" / "Bike_Chest_Start_Cleaned.xlsx"

X_ppg_b01, X_fusion_b01, y_b01, meta_b01 = preprocess_session(
    b01_path,
    sheet_name="Bike_Data",
    activity="Bike",
    session_id="Bike_01",
)

save_session_qc(
    b01_path,
    "Bike_Data",
    "Bike_01",
    X_ppg_b01,
    X_fusion_b01,
    y_b01,
    meta_b01,
    FIGURES / "qc" / "bike",
)

### Bike_02

In [ ]:
b02_path = DATA_RAW / "bike" / "Bike_02_Chest_Start_Cleaned.xlsx"

X_ppg_b02, X_fusion_b02, y_b02, meta_b02 = preprocess_session(
    b02_path,
    sheet_name="Bike_Data",
    activity="Bike",
    session_id="Bike_02",
)

save_session_qc(
    b02_path,
    "Bike_Data",
    "Bike_02",
    X_ppg_b02,
    X_fusion_b02,
    y_b02,
    meta_b02,
    FIGURES / "qc" / "bike",
)

### Bike_03

In [ ]:
b03_path = DATA_RAW / "bike" / "Bike_03_Chest_Start_Cleaned.xlsx"

X_ppg_b03, X_fusion_b03, y_b03, meta_b03 = preprocess_session(
    b03_path,
    sheet_name="Bike_Data",
    activity="Bike",
    session_id="Bike_03",
)

save_session_qc(
    b03_path,
    "Bike_Data",
    "Bike_03",
    X_ppg_b03,
    X_fusion_b03,
    y_b03,
    meta_b03,
    FIGURES / "qc" / "bike",
)

### Bike Summary

In [ ]:
bike_summary = pd.DataFrame({
    "Session": ["Bike_01", "Bike_02", "Bike_03"],
    "Windows": [len(y_b01), len(y_b02), len(y_b03)],
    "Mean_HR": [np.mean(y_b01), np.mean(y_b02), np.mean(y_b03)],
    "Median_HR": [np.median(y_b01), np.median(y_b02), np.median(y_b03)],
    "Min_HR": [np.min(y_b01), np.min(y_b02), np.min(y_b03)],
    "Max_HR": [np.max(y_b01), np.max(y_b02), np.max(y_b03)],
    "Std_HR": [np.std(y_b01), np.std(y_b02), np.std(y_b03)],
}).round(2)

bike_summary.to_csv(METRICS / "bike_summary.csv", index=False)
bike_summary

## 5. Walking

Current available session: `Walking_01`.

When new Walking sessions are collected, duplicate this section and update the file name and session ID.

In [ ]:
w01_path = DATA_RAW / "walking" / "Walk_Chest_Start_Cleaned-1.xlsx"

X_ppg_w01, X_fusion_w01, y_w01, meta_w01 = preprocess_session(
    w01_path,
    sheet_name="Walk_Data",
    activity="Walking",
    session_id="Walking_01",
)

save_session_qc(
    w01_path,
    "Walk_Data",
    "Walking_01",
    X_ppg_w01,
    X_fusion_w01,
    y_w01,
    meta_w01,
    FIGURES / "qc" / "walking",
)

## 6. Food Preparation

Current available session: `Food_Preparation_01`.

In [ ]:
f01_path = DATA_RAW / "food_preparation" / "Food_Preparation_01_Chest_Start_Cleaned.xlsx"

X_ppg_f01, X_fusion_f01, y_f01, meta_f01 = preprocess_session(
    f01_path,
    sheet_name="Food_Preparation_Data",
    activity="Food_Preparation",
    session_id="Food_Preparation_01",
)

save_session_qc(
    f01_path,
    "Food_Preparation_Data",
    "Food_Preparation_01",
    X_ppg_f01,
    X_fusion_f01,
    y_f01,
    meta_f01,
    FIGURES / "qc" / "food_preparation",
)

## 7. Writing

No finalized Writing session has been added yet.

After collecting it, use the same `preprocess_session(...)` + `save_session_qc(...)` pattern.

## 8. Eating

No finalized Eating session has been added yet.

After collecting it, use the same `preprocess_session(...)` + `save_session_qc(...)` pattern.

## 9. Sitting vs Bike Comparison

In [ ]:
sitting_hr = np.concatenate([y_s01, y_s02, y_s03])
bike_hr = np.concatenate([y_b01, y_b02, y_b03])

comparison_summary = pd.DataFrame({
    "Activity": ["Sitting", "Bike"],
    "Windows": [len(sitting_hr), len(bike_hr)],
    "Mean_HR": [np.mean(sitting_hr), np.mean(bike_hr)],
    "Median_HR": [np.median(sitting_hr), np.median(bike_hr)],
    "Min_HR": [np.min(sitting_hr), np.min(bike_hr)],
    "Max_HR": [np.max(sitting_hr), np.max(bike_hr)],
    "Std_HR": [np.std(sitting_hr), np.std(bike_hr)],
}).round(2)

comparison_summary.to_csv(
    METRICS / "sitting_vs_bike_summary.csv",
    index=False,
)

comparison_summary

In [ ]:
plt.figure(figsize=(9, 6))

plt.boxplot(
    [sitting_hr, bike_hr],
    tick_labels=["Sitting", "Bike"],
    widths=0.55,
    showfliers=True,
)

plt.title(
    "Chest HR Distribution - Sitting vs Bike",
    fontsize=16,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Activity")
plt.ylabel("Chest HR (BPM)")
plt.grid(axis="y", alpha=0.2, linestyle="--")
plt.tight_layout()

plt.savefig(
    FIGURES / "comparisons" / "sitting_vs_bike_hr_distribution.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

## 10. Current Dataset Summary

In [ ]:
dataset_summary = pd.DataFrame([
    ["Sitting", "Sitting_01", len(y_s01)],
    ["Sitting", "Sitting_02", len(y_s02)],
    ["Sitting", "Sitting_03", len(y_s03)],
    ["Bike", "Bike_01", len(y_b01)],
    ["Bike", "Bike_02", len(y_b02)],
    ["Bike", "Bike_03", len(y_b03)],
    ["Walking", "Walking_01", len(y_w01)],
    ["Food_Preparation", "Food_Preparation_01", len(y_f01)],
], columns=["Activity", "Session", "Valid_Windows"])

dataset_summary.to_csv(
    METRICS / "dataset_summary.csv",
    index=False,
)

dataset_summary

## 11. Next Step

After QC is complete for all collected sessions:

1. Finalize model-ready processed arrays.
2. Create a **session-aware** train/validation/test split.
3. Normalize using training data only.
4. Train the PPG-only 1D CNN baseline.
5. Train the PPG + ACC fusion 1D CNN.
6. Compare MAE, RMSE, R² and activity-wise performance.
7. Keep activity classification as a separate task.